In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from phase import PhaseSolver, PhaseConfig

def correct(frame, mask):
    ys, xs = np.where(mask)
    corr = frame.astype(float).copy()
    for y, x in zip(ys, xs):
        nb  = frame[max(0,y-1):y+2, max(0,x-1):x+2]
        nbm = mask [max(0,y-1):y+2, max(0,x-1):x+2]
        good = nb[~nbm]
        if good.size: corr[y,x] = np.median(good)
    return corr

N = 20

# Exposure 200 ms

In [ ]:
# Read dark frames
data_dark_200 = np.load("../../data/dark_200_20260907_134836.npz")
stack_dark_200 = data_dark_200['stack'][0]

In [ ]:
# Find and correct for hot pixels
frames = stack_dark_200            # (N,H,W), same gain+exposure as your data

m = frames.mean(0)                 # per-pixel mean  -> high dark current
s = frames.std(0)                  # per-pixel std   -> RTS/flicker

def rob(x, k=5):                   # robust threshold, unbiased by outliers
    med = np.median(x)
    mad = np.median(np.abs(x-med))*1.4826
    return np.abs(x-med) >= k*mad

mask = rob(m) | rob(s)             # (H,W) bool, True = hot
print(100*mask.mean(), "% hot")

stack_dark_200_corr = np.stack([correct(stack_dark_200[n], mask) for n in range(len(stack_dark_200))])
stack_dark_200_ref = stack_dark_200_corr.mean(axis=0)

In [ ]:
# Read ref bright frames
data_ref_bright_200 = np.load("../../data/piezo_sample_bright_200_20260907_135232.npz")
stack_ref_bright_200 = data_ref_bright_200['stack'][0][:N]

In [ ]:
# Correct for hot pixels and subtract dark
stack_ref_bright_200_corr = np.stack([correct(stack_ref_bright_200[n], mask) for n in range(len(stack_ref_bright_200))])
stack_ref_bright_200_corr = np.maximum(stack_ref_bright_200_corr - stack_dark_200_ref, 0)

plt.plot(stack_ref_bright_200_corr[0,1000])
plt.plot(stack_ref_bright_200_corr[3,1000])

In [ ]:
# Compute noise
sigma_n = np.sqrt(stack_ref_bright_200_corr*2.6596) / 2.6596    # (N,P), gain=100
sigma_pixel = np.sqrt((sigma_n**2).mean(0))          # (P,)

rms_ref_bright_200 = np.sqrt((sigma_pixel**2).mean())
noise_floor_ref_bright_200 = np.sqrt((sigma_pixel**2).mean()) / stack_ref_bright_200_corr.std()

print(f"RMS noise: {rms_ref_bright_200}")
print(f"Noise floor: {noise_floor_ref_bright_200}")

In [ ]:
iters = 30
tol = 1e-4
refine_iters = 10
refine_tol = 1e-4

results = {}
for degree in (0,1,2,3,4):
    cfg_d = PhaseConfig.from_yaml("phase_config.yaml")
    cfg_d.use_alpha = True
    cfg_d.gain_mode = "joint"
    cfg_d.precise_reduce = False
    cfg_d.method_kwargs.update(iters=iters, tol=tol, degree=degree,
                                refine_iters=refine_iters, refine_tol=refine_tol)
    solver_d = PhaseSolver(cfg_d)
    solver_d.fit(stack_ref_bright_200_corr)
    results[degree] = solver_d
    mp = solver_d.method_param_
    print(f"degree={degree} converged={mp.refine_converged}  reconstruction_error={solver_d.reconstruction_error_:.4f}  "
          f"kappa_fit={mp.kappa_fit:.3g}  refine_iters_run={mp.refine_iters_run}  "
          f"rms_frac_history={[f'{x:.5f}' for x in mp.rms_frac_history]}")

In [ ]:
degree_arr = np.array([0, 1, 2, 3, 4])
plt.plot(degree_arr, [results[deg].reconstruction_error_ for deg in degree_arr], '.-')
plt.hlines(rms_ref_bright_200, 0, 4, color='k', linestyles='--')
plt.text(0.5,8.5, "Noise floor")
plt.xlabel('Polynomial order')
plt.ylabel('Reconstruction error, RMS')

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(121)
plt.plot(results[0].result_.a[1000], label="aia")
#plt.plot(results[1].result_.a[1000], label="first order")
plt.plot(results[2].result_.a[1000], label="second order")
plt.grid()
plt.legend()
plt.xlim([0, 600])
plt.ylim([170, 220])
plt.xlabel('Pixels')
plt.ylabel('a')

plt.subplot(122)
plt.plot(results[0].result_.b[1000])
#plt.plot(results[1].result_.b[1000])
plt.plot(results[2].result_.b[1000])
plt.grid()
plt.xlabel('Pixels')
plt.ylabel('b')

plt.xlim([0, 600])
plt.ylim([100, 160])

plt.tight_layout()

In [ ]:
num_range = np.arange(start=5, stop=20)
iters = 30
tol = 1e-4
refine_iters = 20
refine_tol = 1e-4

cfg_d = PhaseConfig.from_yaml("phase_config.yaml")
cfg_d.use_alpha = True
cfg_d.gain_mode = "joint"
cfg_d.method_kwargs.update(iters=iters, tol=tol, degree=2,
                            refine_iters=refine_iters, refine_tol=refine_tol)

results = {}
for num in num_range:
    solver_d = PhaseSolver(cfg_d)
    solver_d.fit(stack_ref_bright_200_corr[:num])
    results[num] = solver_d
    mp = solver_d.method_param_
    print(f"num={num} converged={mp.refine_converged}  reconstruction_error={solver_d.reconstruction_error_:.4f}  "
            f"kappa_fit={mp.kappa_fit:.3g}  refine_iters_run={mp.refine_iters_run}  "
            f"rms_frac_history={[f'{x:.5f}' for x in mp.rms_frac_history]}")

In [ ]:
plt.plot(num_range+1, [results[num].method_param_.rms_frac for num in num_range])
plt.hlines(noise_floor_ref_bright_200, 5, 20, color='k', linestyles='--')
plt.text(8,0.086, "Noise floor")

In [ ]:
from phase.methods.step_field import _poly_basis

degree = 2                                        # pick which results[...] fit to inspect
r = results[degree].result_
mp = results[degree].method_param_
a_, b_, phi_ = r.a, r.b, r.phi
u_ = b_ * np.cos(phi_)
v_ = -b_ * np.sin(phi_)

i = a_.size // 2                                  # a pixel index (flat)
row, col = np.unravel_index(i, a_.shape)
u_i, v_i = u_[row, col], v_[row, col]
b_i = np.hypot(u_i, v_i)

# The degree-2 model's actual per-frame step AT THIS PIXEL, not the plain
# piston delta -- delta_n(x,y) = delta_n + coeffs[:,n] @ basis(x,y)
# (docs/step_field_residuals.md Eq. T1), evaluated at (row, col) only (no
# need to build the full (N,H,W) field for one pixel).
basis = _poly_basis(*a_.shape, degree, np)        # (J, P)
basis_i = basis[:, i]                             # (J,) -- this pixel's basis values
delta_eff = r.delta + mp.coeffs.T @ basis_i        # (N,) -- this pixel's true step

xn = u_i * np.cos(delta_eff) + v_i * np.sin(delta_eff)   # fitted x_n, degree-2 model
yn = u_i * np.sin(delta_eff) - v_i * np.cos(delta_eff)   # quadrature

# Raw experimental x_n at the same pixel, alpha/g-corrected the same way the
# model is (I_n = alpha_n*(a + g_n*(u*cos(delta_n)+v*sin(delta_n)))) -- paired
# with the *model's* yn, since a single-channel intensity sample only gives
# one real number per frame, not an independent (x,y) pair. Points straying
# off the fitted circle are frames where the raw data disagrees with what
# the degree-2 fit predicts -- this is the actual "how good is my model"
# check, now that the reference uses the fitted step field, not the piston.
xn_raw = (stack_ref_bright_200_corr[:, row, col] / r.alpha - a_[row, col]) / r.g

theta = np.linspace(0, 2 * np.pi, 400)
plt.plot(b_i * np.cos(theta), b_i * np.sin(theta), '-', color='0.7', lw=1,
         label='ideal circle (radius b)')
plt.plot(xn, yn, 'o', label=f'fitted (degree={degree})')
plt.plot(xn_raw, yn, 'x', label='raw data')
plt.gca().set_aspect('equal')
plt.xlabel('x_n')
plt.ylabel('y_n')
plt.title(f'Lissajous ellipse at one pixel (degree={degree})')
plt.legend()

In [ ]:
from phase.backend import wrap

xr, yr = xn_raw, yn          # yr still comes from the model -- see the note in the cell above
th = np.arctan2(yr, xr)
rad = np.hypot(xr, yr)

print("radial scatter (amp err):", np.std(rad) / b_i)                # alpha/g miscalibration, noise
print("angular scatter (delta err):", np.std(wrap(th - delta_eff)))  # phase-step (delta_n) error

# Exposure 400 ms

In [ ]:
# Read dark frames
data_dark_400 = np.load("../../data/dark_400_20260907_134803.npz")
stack_dark_400 = data_dark_400['stack'][0]

In [ ]:
# Find and correct for hot pixels
frames = stack_dark_400            # (N,H,W), same gain+exposure as your data

m = frames.mean(0)                 # per-pixel mean  -> high dark current
s = frames.std(0)                  # per-pixel std   -> RTS/flicker

def rob(x, k=5):                   # robust threshold, unbiased by outliers
    med = np.median(x)
    mad = np.median(np.abs(x-med))*1.4826
    return np.abs(x-med) >= k*mad

mask = rob(m) | rob(s)             # (H,W) bool, True = hot
print(100*mask.mean(), "% hot")

stack_dark_400_corr = np.stack([correct(stack_dark_400[n], mask) for n in range(len(stack_dark_400))])
stack_dark_400_ref = stack_dark_400_corr.mean(axis=0)

In [ ]:
# Read ref bright frames
data_ref_bright_400 = np.load("../../data/piezo_sample_bright_400_20260907_135149.npz")
stack_ref_bright_400 = data_ref_bright_400['stack'][0]

In [ ]:
# Correct for hot pixels and subtract dark
stack_ref_bright_400_corr = np.stack([correct(stack_ref_bright_400[n], mask) for n in range(len(stack_ref_bright_400))])
stack_ref_bright_400_corr = np.maximum(stack_ref_bright_400_corr - stack_dark_400_ref, 0)

plt.plot(stack_ref_bright_400_corr[0,1000])
plt.plot(stack_ref_bright_400_corr[1,1000])

In [ ]:
# Compute noise
sigma_n = np.sqrt(stack_ref_bright_400_corr*2.6596) / 2.6596    # (N,P), gain=100
sigma_pixel = np.sqrt((sigma_n**2).mean(0))          # (P,)

rms_ref_bright_400 = np.sqrt((sigma_pixel**2).mean())
noise_floor_ref_bright_400 = np.sqrt((sigma_pixel**2).mean()) / stack_ref_bright_400_corr.std()

print(f"RMS noise: {rms_ref_bright_400}")
print(f"Noise floor: {noise_floor_ref_bright_400}")

In [ ]:
iters = 30
tol = 1e-4
refine_iters = 20
refine_tol = 1e-4

results = {}
for degree in (0, 1, 2):
    cfg_d = PhaseConfig.from_yaml("phase_config.yaml")
    cfg_d.use_alpha = True
    cfg_d.gain_mode = "joint"
    cfg_d.method_kwargs.update(iters=iters, tol=tol, degree=degree,
                                refine_iters=refine_iters, refine_tol=refine_tol)
    solver_d = PhaseSolver(cfg_d)
    solver_d.fit(stack_ref_bright_400_corr)
    results[degree] = solver_d
    mp = solver_d.method_param_
    print(f"degree={degree} converged={mp.refine_converged}  reconstruction_error={solver_d.reconstruction_error_:.4f}  "
          f"kappa_fit={mp.kappa_fit:.3g}  refine_iters_run={mp.refine_iters_run}  "
          f"rms_frac_history={[f'{x:.5f}' for x in mp.rms_frac_history]}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(121)
plt.plot(results[0].result_.a[1000], label="aia")
plt.plot(results[2].result_.a[1000], label="second order")
plt.grid()
plt.legend()

plt.subplot(122)
plt.plot(results[0].result_.b[1000])
plt.plot(results[2].result_.b[1000])
plt.grid()

plt.tight_layout()

In [ ]:
num_range = np.arange(start=5, stop=20)
iters = 30
tol = 1e-4
refine_iters = 20
refine_tol = 1e-4

cfg_d = PhaseConfig.from_yaml("phase_config.yaml")
cfg_d.use_alpha = True
cfg_d.gain_mode = "joint"
cfg_d.method_kwargs.update(iters=iters, tol=tol, degree=2,
                            refine_iters=refine_iters, refine_tol=refine_tol)

results = {}
for num in num_range:
    solver_d = PhaseSolver(cfg_d)
    solver_d.fit(stack_ref_bright_400_corr[:num])
    results[num] = solver_d
    mp = solver_d.method_param_
    print(f"num={num} converged={mp.refine_converged}  reconstruction_error={solver_d.reconstruction_error_:.4f}  "
            f"kappa_fit={mp.kappa_fit:.3g}  refine_iters_run={mp.refine_iters_run}  "
            f"rms_frac_history={[f'{x:.5f}' for x in mp.rms_frac_history]}")

In [ ]:
plt.plot(num_range+1, [results[num].method_param_.rms_frac for num in num_range])
plt.hlines(noise_floor_ref_bright_400, 5, 20, color='k', linestyles='--')
plt.text(8,0.059, "Noise floor")

In [ ]:
from phase.methods.step_field import _poly_basis

degree = 3                                        # pick which results[...] fit to inspect
r = results[degree].result_
mp = results[degree].method_param_
a_, b_, phi_ = r.a, r.b, r.phi
u_ = b_ * np.cos(phi_)
v_ = -b_ * np.sin(phi_)

i = a_.size // 2                                  # a pixel index (flat)
row, col = np.unravel_index(i, a_.shape)
u_i, v_i = u_[row, col], v_[row, col]
b_i = np.hypot(u_i, v_i)

# The degree-2 model's actual per-frame step AT THIS PIXEL, not the plain
# piston delta -- delta_n(x,y) = delta_n + coeffs[:,n] @ basis(x,y)
# (docs/step_field_residuals.md Eq. T1), evaluated at (row, col) only (no
# need to build the full (N,H,W) field for one pixel).
basis = _poly_basis(*a_.shape, degree, np)        # (J, P)
basis_i = basis[:, i]                             # (J,) -- this pixel's basis values
delta_eff = r.delta + mp.coeffs.T @ basis_i        # (N,) -- this pixel's true step

xn = u_i * np.cos(delta_eff) + v_i * np.sin(delta_eff)   # fitted x_n, degree-2 model
yn = u_i * np.sin(delta_eff) - v_i * np.cos(delta_eff)   # quadrature

# Raw experimental x_n at the same pixel, alpha/g-corrected the same way the
# model is (I_n = alpha_n*(a + g_n*(u*cos(delta_n)+v*sin(delta_n)))) -- paired
# with the *model's* yn, since a single-channel intensity sample only gives
# one real number per frame, not an independent (x,y) pair. Points straying
# off the fitted circle are frames where the raw data disagrees with what
# the degree-2 fit predicts -- this is the actual "how good is my model"
# check, now that the reference uses the fitted step field, not the piston.
xn_raw = (stack_ref_bright_400_corr[:, row, col] / r.alpha - a_[row, col]) / r.g

theta = np.linspace(0, 2 * np.pi, 400)
plt.plot(b_i * np.cos(theta), b_i * np.sin(theta), '-', color='0.7', lw=1,
         label='ideal circle (radius b)')
plt.plot(xn, yn, 'o', label=f'fitted (degree={degree})')
plt.plot(xn_raw, yn, 'x', label='raw data')
plt.gca().set_aspect('equal')
plt.xlabel('x_n')
plt.ylabel('y_n')
plt.title(f'Lissajous ellipse at one pixel (degree={degree})')
plt.legend()

In [ ]:
xr, yr = xn_raw, yn          # yr still comes from the model -- see the note in the cell above
th = np.arctan2(yr, xr)
rad = np.hypot(xr, yr)

print("radial scatter (amp err):", np.std(rad) / b_i)                # alpha/g miscalibration, noise
print("angular scatter (delta err):", np.std(wrap(th - delta_eff)))  # phase-step (delta_n) error

In [ ]:
plt.imshow(results[3].result_.b/results[3].result_.a)
# plt.xlim([500, 1500])
# plt.ylim([1750, 1250])

In [ ]:
plt.plot(results[3].result_.delta, '.-')

In [ ]:
results[3].method_param_

In [ ]:
plt.plot(stack_ref_bright_400_corr[0,1000])
plt.plot(stack_ref_bright_400_corr[5,1000])